In [ ]:
import os

import numpy as np
import torch
import pandas as pd

from src.cann.train import ACT_DICT
from src.cann.models.base import ActNN
from src.cann.models.polykan import PolyKAN
from src.cann.train import seed_all

import matplotlib.pyplot as plt

In [ ]:
def numerical_gradient(activation: callable, x: torch.Tensor) -> torch.Tensor:

    with torch.no_grad():
        my_grad = 0 * x
        h = 1e-2
    
        #assumption here is 1d array
        for idx, xi in enumerate(x):
            fplus = activation(xi[None]+h)
            fneg = activation(xi[None]-h)
    
            my_grad[idx] = (fplus - fneg) / (2*h)

    return my_grad
    
def compute_gradient(activation: callable, x: torch.Tensor=None) -> torch.Tensor:
    if x is None:
        x = torch.arange(-8, 8, 0.01)
        
    my_grad = numerical_gradient(activation, x)

    return my_grad
        
def check_grad_and_monotonic(activation: callable, x: torch.Tensor) -> str:

    my_grad = compute_gradient(activation, x)
    
    # strictly monotonic - always increasing/decreasing
    # weakly monotonic - contains areas of zero gradient
    # non-monotonic - slope switch from + to - somewhere

    monotonic = ""
    for index in range(my_grad.shape[1]):
        
        if torch.sign(my_grad[:,index,:,:]).min() == torch.sign(my_grad[:,index,:,:]).max():
            # if slope is always negative or always positive
            # function is strictly monotonic /
            monotonic += "strictly"
        elif torch.abs(torch.sign(my_grad[:,index,:,:]).min()) + torch.abs(torch.sign(my_grad[:,index,:,:]).max()) == 1:
            # if slope is always negative or 0, or always positive or 0, 
            # function is weakly monotonic _/
            monotonic += "weakly"
        elif torch.abs(torch.sign(my_grad[:,index,:,:]).min()) + torch.abs(torch.sign(my_grad[:,index,:,:]).max()) == 2:
            # function includes positive and negative slope
            # non-monotonic \/
            monotonic += "non"
        else:
            assert False, "Error: unreachable scope reached"

        
            
    return my_grad, monotonic

In [ ]:
c1 = plt.get_cmap("magma")(32)
c2 = plt.get_cmap("magma")(192)
wd = 1
for act_key in ACT_DICT.keys():
    act = ACT_DICT[act_key](wd=wd)

    x = torch.arange(-8,8,0.01)[:,None,None,None]
    
    my_grad, mono = check_grad_and_monotonic(act, x)

    print(f"{act_key},\n\t{mono} monotonic, {my_grad.min().item():.3e}, {my_grad.max().item():.3e}, {my_grad.mean().item():.3e}")
    
    with torch.no_grad():
        fig, ax = plt.subplots(1,1)
        ax.plot(x[:,0,0,0], my_grad[:,0,0,0],color=c1, label="f'(x)")
        ax.set_ylabel("f'(x)")
        ax2 = ax.twinx()
        ax2.plot(x[:,0,0,0], act(x)[:,0,0,0],"--", color=c2, label="f'(x)")
        ax2.set_xlabel("x")
        ax2.set_ylabel("f(x)")
        fig.suptitle(f"{act_key} activation")
        fig.legend()
        plt.savefig(f"activ_{act_key}.png")
        plt.show()

seed_all(13)
for ii in range(10):
    model = PolyKAN()
    act_key = "polynomial activation"
    
    act = model.activations[0]

    x = torch.arange(-8,8,0.01).reshape(-1,1,1,1)* torch.ones(1,2,1,1)
    
    
    my_grad, mono = check_grad_and_monotonic(act, x)

    print(f"{act_key},\n\t{mono} monotonic, {my_grad.min().item():.3e}, {my_grad.max().item():.3e}, {my_grad.mean().item():.3e}")
    
    with torch.no_grad():
        fig, ax = plt.subplots(1,1)
        ax.plot(x[:,0,0,0], my_grad[:,0,0,0],color=c1)
        ax.plot(x[:,1,0,0], my_grad[:,1,0,0],color=c1, label="f'(x)")
        ax.set_ylabel("f'(x)")
        ax2 = ax.twinx()
        ax2.plot(x[:,0,0,0], act(x)[:,0,0,0], "--", color=c2 )
        ax2.plot(x[:,0,0,0], act(x)[:,1,0,0], "--", color=c2, label="f(x)")
        ax2.set_xlabel("x")
        ax2.set_ylabel("f(x)")
        fig.suptitle("polynomial activation")
        fig.legend()
        plt.savefig(f"activ_{act_key}.png")
        plt.show()

seed_all(13)
print(x.shape), my_grad.shape, act(x).shape

plt.figure()
plt.plot(x[:,0,0,0], my_grad[:,0,0,0])
plt.plot(x[:,0,0,0], my_grad[:,0,0,0])
plt.show()

In [ ]:
root_dir = "../results"

dir_list = [\
    "poly_no_knockout_l1_1",\
    "poly_weight_knockout_neur_l1_1"]

param_index = -1
for fn in dir_list:
    filepath = os.path.join(root_dir, fn, "exp.csv")

    df = pd.read_csv(filepath)
    del df["Unnamed: 0"]
    
    for run_index in range(len(df)):
        
        model_width = df["model_width"][run_index].item()
        model_depth = df["model_depth"][run_index].item()
        activation_name = df["activation_name"][run_index].lower()
        
        run_df = pd.read_csv(os.path.join("..", df["run_filename"][run_index]))
        del run_df["Unnamed: 0"]
    
        parameters = np.load(os.path.join("..", run_df["parameters_filename"][0]))
        
        if activation_name == "polykan":
            model = PolyKAN(width=model_width, depth=model_depth)
        else:
            model = ActNN(width=model_width, depth=model_depth, activation=ACT_DICT[activation_name])
    
        model.set_parameters(parameters[param_index])

        for act_index, act in enumerate(model.activations):
            act = model.activations[act_index]
    
             
            if act_index:
                x = torch.arange(-8,8,0.01).reshape(-1,1,1,1)* torch.ones(1,1,1,1)
            else:
                x = torch.arange(-8,8,0.01).reshape(-1,1,1,1)* torch.ones(1,2,1,1)
            print(x.shape)
            
            my_grad, mono = check_grad_and_monotonic(act, x)
    
            print(f"{fn} {act_index},\n\t{mono} monotonic, {my_grad.min().item():.3e}, {my_grad.max().item():.3e}, {my_grad.mean().item():.3e}")
            with torch.no_grad():
                plt.figure()
                if act_index:
                    with torch.no_grad():
                        fig, ax = plt.subplots(1,1)
                        ax.plot(x[:,0,0,0], my_grad[:,0,0,0],color=c1, label="f'(x)")
                        ax.set_ylabel("f'(x)")
                        ax2 = ax.twinx()
                        ax2.plot(x[:,0,0,0], act(x)[:,0,0,0],"--", color=c2, label="f'(x)")
                        ax2.set_xlabel("x")
                        ax2.set_ylabel("f(x)")
                        fig.suptitle(f"{act_key} activation")
                        fig.legend()
                        plt.show()
                else:
                    with torch.no_grad():
                        fig, ax = plt.subplots(1,1)
                        ax.plot(x[:,0,0,0], my_grad[:,0,0,0],color=c1)
                        ax.plot(x[:,1,0,0], my_grad[:,1,0,0],color=c1, label="f'(x)")
                        ax.set_ylabel("f'(x)")
                        ax2 = ax.twinx()
                        ax2.plot(x[:,0,0,0], act(x)[:,0,0,0], "--", color=c2 )
                        ax2.plot(x[:,0,0,0], act(x)[:,1,0,0], "--", color=c2, label="f(x)")
                        ax2.set_xlabel("x")
                        ax2.set_ylabel("f(x)")
                        fig.suptitle("polynomial activation")
                        fig.legend()
                        plt.show()
                    
                    
                plt.show()
        

In [ ]:
root_dir = "../results"

dir_list = [
    "prelu_no_knockout_l1_1",\
    "prelu_weight_knockout_neur_l1_1"
    ]

fig_count = 0
param_index = -1
for fn in dir_list:
    filepath = os.path.join(root_dir, fn, "exp.csv")

    df = pd.read_csv(filepath)
    del df["Unnamed: 0"]
    
    for run_index in range(len(df)):
        
        model_width = df["model_width"][run_index].item()
        model_depth = df["model_depth"][run_index].item()
        activation_name = df["activation_name"][run_index].lower()
        
        run_df = pd.read_csv(os.path.join("..", df["run_filename"][run_index]))
        del run_df["Unnamed: 0"]
    
        parameters = np.load(os.path.join("..", run_df["parameters_filename"][0]))
        
        if activation_name == "polykan":
            model = PolyKAN(width=model_width, depth=model_depth)
        else:
            model = ActNN(width=model_width, depth=model_depth, activation=ACT_DICT[activation_name])
    
        model.set_parameters(parameters[param_index])

        for act_index, act in enumerate(model.activations):
            act = model.activations[act_index]
    
             
            if act_index:
                x = torch.arange(-8,8,0.01).reshape(-1,1,1,1)* torch.ones(1,1,1,1)
            else:
                x = torch.arange(-8,8,0.01).reshape(-1,1,1,1)* torch.ones(1,2,1,1)
            print(x.shape)
            
            my_grad, mono = check_grad_and_monotonic(act, x)
    
            print(f"{fn} {act_index},\n\t{mono} monotonic, {my_grad.min().item():.3e}, {my_grad.max().item():.3e}, {my_grad.mean().item():.3e}")
            with torch.no_grad():
                plt.figure()
                if act_index:
                    with torch.no_grad():
                        fig, ax = plt.subplots(1,1)
                        ax.plot(x[:,0,0,0], my_grad[:,0,0,0],color=c1, label="f'(x)")
                        ax.set_ylabel("f'(x)")
                        ax2 = ax.twinx()
                        ax2.plot(x[:,0,0,0], act(x)[:,0,0,0],"--", color=c2, label="f(x)")
                        ax2.set_xlabel("x")
                        ax2.set_ylabel("f(x)")
                        fig.suptitle(f"{act_key} activation")
                        fig.legend()
                        #plt.savefig("act_function{act_key}{fig_count}.png")
                else:
                    with torch.no_grad():
                        fig, ax = plt.subplots(1,1)
                        ax.plot(x[:,0,0,0], my_grad[:,0,0,0],color=c1, label="f'(x)")
                        ax.plot(x[:,1,0,0], my_grad[:,1,0,0],color=c1, label="_f'(x)")
                        ax.set_ylabel("f'(x)")
                        ax2 = ax.twinx()
                        ax2.plot(x[:,0,0,0], act(x)[:,0,0,0], "--", color=c2, label="f(x)" )
                        ax2.plot(x[:,0,0,0], act(x)[:,1,0,0], "--", color=c2, label="_f(x)")
                        ax2.set_xlabel("x")
                        ax2.set_ylabel("f(x)")
                        fig.suptitle("polynomial activation")
                        fig.legend()
                        
                        #plt.savefig("act_function{act_key}{fig_count}.png")
                    
                fig_count += 1
                plt.show()
        